# 05b · Busca de hiperparâmetros

Escolhe, para cada arquitetura, a configuração com a menor perda de validação em uma grade de quatro configurações. Variam a largura, o número de camadas e a taxa de aprendizado; o resto do protocolo do notebook 05 se mantém.

A única mudança no protocolo é o ***gradient clipping*** (norma máxima 1,0), aplicado às três arquiteturas para estabilizar a BiLSTM sem tratá-la de forma diferente das demais.

O conjunto de teste não é usado aqui. Precisa de GPU (T4); a busca completa levou cerca de 20 minutos. Cada configuração concluída fica registrada, e as já feitas são puladas em uma nova execução.

## 1. Configuração

In [ ]:
import os, json, time, random, shutil
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR   = '/content/drive/MyDrive/TCC-Exoplanetas'
MODELS_DIR = f'{DATA_DIR}/modelos'
BUSCA_DIR  = f'{MODELS_DIR}/busca'
os.makedirs(BUSCA_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('*** ATENCAO: ative Runtime -> T4 GPU antes de rodar a busca. ***')

Mounted at /content/drive
device: cuda


In [ ]:
SEED = 42

def fixar_sementes(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# --- CONSTANTES DO PROTOCOLO (iguais para todas as arquiteturas) ---
BATCH_SIZE = 64
MAX_EPOCAS = 50
PACIENCIA  = 8
CLIP_NORMA = 1.0        # NOVO: gradient clipping, igual para as tres
print(f'Protocolo fixo: batch {BATCH_SIZE} | max epocas {MAX_EPOCAS} | paciencia {PACIENCIA} | clip {CLIP_NORMA}')

Protocolo fixo: batch 64 | max epocas 50 | paciencia 8 | clip 1.0


In [ ]:
d = np.load(f'{DATA_DIR}/dataset.npz')

class VisoesDataset(Dataset):
    def __init__(self, Xg, Xl, y):
        self.Xg = torch.from_numpy(Xg).float().unsqueeze(1)
        self.Xl = torch.from_numpy(Xl).float().unsqueeze(1)
        self.y  = torch.from_numpy(y).float()
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.Xg[i], self.Xl[i], self.y[i]

ds_train = VisoesDataset(d['Xg_train'], d['Xl_train'], d['y_train'])
ds_val   = VisoesDataset(d['Xg_val'],   d['Xl_val'],   d['y_val'])
print(f'treino {len(ds_train)} | val {len(ds_val)}')

treino 6312 | val 683


## 2. Arquiteturas

As mesmas do notebook 04, com os hiperparâmetros expostos como argumentos.

In [ ]:
class DoisRamos(nn.Module):
    def __init__(self, fabrica_ramo, p_dropout=0.3):
        super().__init__()
        self.ramo_global = fabrica_ramo()
        self.ramo_local  = fabrica_ramo()
        d_ = self.ramo_global.out_dim
        self.cabeca = nn.Sequential(
            nn.Linear(2 * d_, 64), nn.ReLU(), nn.Dropout(p_dropout), nn.Linear(64, 1))
    def forward(self, g, l):
        return self.cabeca(torch.cat([self.ramo_global(g), self.ramo_local(l)], 1)).squeeze(-1)


class RamoCNN(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        c1, c2, c3 = base, base * 2, base * 4
        self.net = nn.Sequential(
            nn.Conv1d(1, c1, 5, padding=2), nn.ReLU(), nn.MaxPool1d(5, 2),
            nn.Conv1d(c1, c2, 5, padding=2), nn.ReLU(), nn.MaxPool1d(5, 2),
            nn.Conv1d(c2, c3, 5, padding=2), nn.ReLU(),
            nn.AdaptiveMaxPool1d(1))
        self.out_dim = c3
    def forward(self, x): return self.net(x).squeeze(-1)


class RamoBiLSTM(nn.Module):
    def __init__(self, hidden=64, camadas=2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, num_layers=camadas, batch_first=True,
                            bidirectional=True, dropout=0.1 if camadas > 1 else 0.0)
        self.out_dim = 2 * hidden
    def forward(self, x):
        saida, _ = self.lstm(x.transpose(1, 2))
        return saida.mean(dim=1)


def codificacao_posicional(comprimento, dim, device):
    pos = torch.arange(comprimento, device=device).unsqueeze(1).float()
    i   = torch.arange(dim, device=device).unsqueeze(0).float()
    ang = pos / torch.pow(torch.tensor(10000.0, device=device), (2 * (i // 2)) / dim)
    pe = torch.zeros(comprimento, dim, device=device)
    pe[:, 0::2] = torch.sin(ang[:, 0::2]); pe[:, 1::2] = torch.cos(ang[:, 1::2])
    return pe.unsqueeze(0)


class RamoTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4, camadas=2, stride=8):
        super().__init__()
        self.embed = nn.Conv1d(1, d_model, kernel_size=2 * stride + 1, stride=stride, padding=stride)
        camada = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128,
                                            dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(camada, camadas)
        self.out_dim = d_model
    def forward(self, x):
        z = self.embed(x).transpose(1, 2)
        z = z + codificacao_posicional(z.size(1), z.size(2), z.device)
        return self.encoder(z).mean(dim=1)


CLASSES = {'cnn1d': RamoCNN, 'bilstm': RamoBiLSTM, 'transformer': RamoTransformer}
NOMES   = {'cnn1d': 'CNN 1D', 'bilstm': 'BiLSTM', 'transformer': 'Transformer'}

## 3. Grade de busca

Quatro configurações por arquitetura, combinando duas taxas de aprendizado com variações de largura e, na BiLSTM e no Transformer, de número de camadas. Para a BiLSTM, a grade inclui modelos e taxas de aprendizado menores, por causa da instabilidade observada no notebook 05.

In [ ]:
GRADES = {
    'cnn1d': [
        {'lr': 1e-3, 'ramo': {'base': 16}},
        {'lr': 3e-4, 'ramo': {'base': 16}},
        {'lr': 1e-3, 'ramo': {'base': 32}},
        {'lr': 3e-4, 'ramo': {'base': 32}},
    ],
    'bilstm': [
        {'lr': 1e-3, 'ramo': {'hidden': 64, 'camadas': 2}},   # config original
        {'lr': 3e-4, 'ramo': {'hidden': 64, 'camadas': 2}},
        {'lr': 3e-4, 'ramo': {'hidden': 32, 'camadas': 1}},
        {'lr': 1e-3, 'ramo': {'hidden': 32, 'camadas': 1}},
    ],
    'transformer': [
        {'lr': 1e-3, 'ramo': {'d_model': 64,  'camadas': 2}}, # config original
        {'lr': 3e-4, 'ramo': {'d_model': 64,  'camadas': 2}},
        {'lr': 1e-3, 'ramo': {'d_model': 128, 'camadas': 2}},
        {'lr': 3e-4, 'ramo': {'d_model': 64,  'camadas': 4}},
    ],
}
print('Total de treinos:', sum(len(v) for v in GRADES.values()))

Total de treinos: 12


## 4. Treinamento de uma configuração

Igual ao laço do notebook 05, com o *gradient clipping* aplicado antes de cada passo do otimizador.

In [ ]:
@torch.no_grad()
def avaliar(modelo, loader, criterio):
    modelo.eval()
    perdas, probs, alvos = [], [], []
    for g, l, y in loader:
        g, l, y = g.to(device), l.to(device), y.to(device)
        logits = modelo(g, l)
        perdas.append(criterio(logits, y).item() * len(y))
        probs.append(torch.sigmoid(logits).cpu().numpy()); alvos.append(y.cpu().numpy())
    probs = np.concatenate(probs); alvos = np.concatenate(alvos)
    return sum(perdas) / len(alvos), roc_auc_score(alvos, probs)


def treinar_config(slug, cfg, chave):
    fixar_sementes()
    ger = torch.Generator(); ger.manual_seed(SEED)
    dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, generator=ger)
    dl_val   = DataLoader(ds_val, batch_size=BATCH_SIZE)

    Classe = CLASSES[slug]
    modelo = DoisRamos(lambda: Classe(**cfg['ramo'])).to(device)
    criterio = nn.BCEWithLogitsLoss()
    otim = torch.optim.Adam(modelo.parameters(), lr=cfg['lr'])
    n_par = sum(p.numel() for p in modelo.parameters())

    melhor, espera, melhor_estado, melhor_auc = float('inf'), 0, None, 0.0
    t0 = time.time()
    for epoca in range(1, MAX_EPOCAS + 1):
        modelo.train()
        for g, l, y in dl_train:
            g, l, y = g.to(device), l.to(device), y.to(device)
            otim.zero_grad()
            criterio(modelo(g, l), y).backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), CLIP_NORMA)   # <-- clipping
            otim.step()
        perda_val, auc_val = avaliar(modelo, dl_val, criterio)
        if perda_val < melhor - 1e-5:
            melhor, espera, melhor_auc = perda_val, 0, auc_val
            melhor_estado = {k: v.cpu().clone() for k, v in modelo.state_dict().items()}
        else:
            espera += 1
            if espera >= PACIENCIA:
                break
    dur = time.time() - t0
    torch.save(melhor_estado, f'{BUSCA_DIR}/{chave}.pt')
    return {'slug': slug, 'cfg': cfg, 'melhor_val': melhor, 'auc_val': melhor_auc,
            'parametros': n_par, 'epocas': epoca, 'tempo_s': dur}

## 5. Execução da busca

Os resultados são gravados em `busca_resultados.json` à medida que cada configuração termina.

In [ ]:
RES_PATH = f'{MODELS_DIR}/busca_resultados.json'
resultados = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}

for slug, grade in GRADES.items():
    for i, cfg in enumerate(grade):
        chave = f'{slug}_cfg{i}'
        if chave in resultados:
            print(f'[pulado] {chave}')
            continue
        print(f'treinando {chave}: lr={cfg["lr"]}  {cfg["ramo"]} ...', end=' ', flush=True)
        r = treinar_config(slug, cfg, chave)
        resultados[chave] = r
        json.dump(resultados, open(RES_PATH, 'w'), indent=2)
        print(f"val {r['melhor_val']:.4f}  AUC {r['auc_val']:.4f}  "
              f"({r['epocas']} ep, {r['tempo_s']/60:.1f} min)")

print('\nBusca concluida.')

treinando cnn1d_cfg0: lr=0.001  {'base': 16} ... val 0.2813  AUC 0.9559  (24 ep, 0.3 min)
treinando cnn1d_cfg1: lr=0.0003  {'base': 16} ... val 0.2964  AUC 0.9488  (24 ep, 0.3 min)
treinando cnn1d_cfg2: lr=0.001  {'base': 32} ... val 0.2848  AUC 0.9553  (24 ep, 0.4 min)
treinando cnn1d_cfg3: lr=0.0003  {'base': 32} ... val 0.2838  AUC 0.9518  (20 ep, 0.4 min)
treinando bilstm_cfg0: lr=0.001  {'hidden': 64, 'camadas': 2} ... val 0.3772  AUC 0.9107  (50 ep, 6.1 min)
treinando bilstm_cfg1: lr=0.0003  {'hidden': 64, 'camadas': 2} ... val 0.3486  AUC 0.9234  (50 ep, 6.1 min)
treinando bilstm_cfg2: lr=0.0003  {'hidden': 32, 'camadas': 1} ... val 0.4426  AUC 0.8741  (50 ep, 1.2 min)
treinando bilstm_cfg3: lr=0.001  {'hidden': 32, 'camadas': 1} ... val 0.3330  AUC 0.9312  (46 ep, 1.1 min)
treinando transformer_cfg0: lr=0.001  {'d_model': 64, 'camadas': 2} ... val 0.3564  AUC 0.9375  (24 ep, 1.1 min)
treinando transformer_cfg1: lr=0.0003  {'d_model': 64, 'camadas': 2} ... val 0.2946  AUC 0.9477

## 6. Resultados da busca

In [ ]:
import pandas as pd
linhas = []
for chave, r in resultados.items():
    linhas.append({
        'arquitetura': NOMES[r['slug']], 'config': chave.split('_')[-1],
        'lr': r['cfg']['lr'], 'ramo': str(r['cfg']['ramo']),
        'val_loss': round(r['melhor_val'], 4), 'AUC_val': round(r['auc_val'], 4),
        'params': r['parametros'], 'epocas': r['epocas'], 'min': round(r['tempo_s'] / 60, 1),
    })
tab = pd.DataFrame(linhas).sort_values(['arquitetura', 'val_loss'])
display(tab)

,arquitetura,config,lr,ramo,val_loss,AUC_val,params,epocas,min
7,BiLSTM,cfg3,0.0010,"{'hidden': 32, 'camadas': 1}",0.3330,0.9312,26241,46,1.1
5,BiLSTM,cfg1,0.0003,"{'hidden': 64, 'camadas': 2}",0.3486,0.9234,283777,50,6.1
4,BiLSTM,cfg0,0.0010,"{'hidden': 64, 'camadas': 2}",0.3772,0.9107,283777,50,6.1
6,BiLSTM,cfg2,0.0003,"{'hidden': 32, 'camadas': 1}",0.4426,0.8741,26241,50,1.2
0,CNN 1D,cfg0,0.0010,{'base': 16},0.2813,0.9559,34305,24,0.3
3,CNN 1D,cfg3,0.0003,{'base': 32},0.2838,0.9518,119681,20,0.4
2,CNN 1D,cfg2,0.0010,{'base': 32},0.2848,0.9553,119681,24,0.4
1,CNN 1D,cfg1,0.0003,{'base': 16},0.2964,0.9488,34305,24,0.3
9,Transformer,cfg1,0.0003,"{'d_model': 64, 'camadas': 2}",0.2946,0.9477,144513,24,1.1
10,Transformer,cfg2,0.0010,"{'d_model': 128, 'camadas': 2}",0.3049,0.9436,419457,13,0.8


## 7. Configuração final

Para cada arquitetura, os pesos da configuração com a menor perda de validação são copiados para `{slug}.pt`, e a configuração é registrada, junto com o protocolo, em `config_final.json`. São as configurações da Tabela 3 do TCC. Os pesos do treinamento inicial ficam preservados como `{slug}_baseline.pt`.

In [ ]:
config_final = {}
for slug in GRADES:
    cands = {k: v for k, v in resultados.items() if v['slug'] == slug}
    melhor_chave = min(cands, key=lambda k: cands[k]['melhor_val'])
    r = cands[melhor_chave]

    destino = f'{MODELS_DIR}/{slug}.pt'
    baseline = f'{MODELS_DIR}/{slug}_baseline.pt'
    if os.path.exists(destino) and not os.path.exists(baseline):
        shutil.copy(destino, baseline)          # preserva a linha de base
    shutil.copy(f'{BUSCA_DIR}/{melhor_chave}.pt', destino)

    config_final[slug] = {'arquitetura': NOMES[slug], 'config_vencedora': melhor_chave,
                          'lr': r['cfg']['lr'], 'ramo': r['cfg']['ramo'],
                          'val_loss': r['melhor_val'], 'auc_val': r['auc_val'],
                          'parametros': r['parametros'], 'epocas': r['epocas'],
                          'tempo_treino_s': r['tempo_s'],
                          'protocolo': {'batch': BATCH_SIZE, 'max_epocas': MAX_EPOCAS,
                                        'paciencia': PACIENCIA, 'clip_norma': CLIP_NORMA,
                                        'otimizador': 'Adam', 'perda': 'BCEWithLogitsLoss',
                                        'seed': SEED}}
    print(f"{NOMES[slug]:12s} -> {melhor_chave}  lr={r['cfg']['lr']}  {r['cfg']['ramo']}  "
          f"val {r['melhor_val']:.4f}  AUC {r['auc_val']:.4f}")

json.dump(config_final, open(f'{MODELS_DIR}/config_final.json', 'w'), indent=2)
print('\nSalvo: config_final.json  (registro de reprodutibilidade)')

CNN 1D       -> cnn1d_cfg0  lr=0.001  {'base': 16}  val 0.2813  AUC 0.9559
BiLSTM       -> bilstm_cfg3  lr=0.001  {'hidden': 32, 'camadas': 1}  val 0.3330  AUC 0.9312
Transformer  -> transformer_cfg1  lr=0.0003  {'d_model': 64, 'camadas': 2}  val 0.2946  AUC 0.9477

Salvo: config_final.json  (registro de reprodutibilidade)


## Próximo passo

O notebook 05c treina as configurações finais com três sementes.